# Amplitūdu pastiprināšana (Amplitude Amplification) 

Amplitūdu pastiprināšana ir vispārinājums Grovera meklēšanai. Tā ļauj palielināt “labā” (atzīmētā) rezultāta varbūtību, ja jau ir pieejams kvantu algoritms $A$, kas risinājumu atrod ar kādu varbūtību $p$.

Ja algoritms $A$ reizēm dod pareizo atbildi, tad ar īpašu refleksiju (spoguļošanas) operatoru palīdzību mēs varam “pagriezt” kvantu stāvokli tā, lai varbūtība iegūt pareizo atbildi kļūtu lielāka.


## Pamatideja

Mums ir unitārs operators $A$, kas no sākuma stāvokļa $|0\rangle$ sagatavo stāvokli $|\psi\rangle = A|0\rangle.$

Šajā stāvoklī daļa amplitūdu ir uz "marķētajiem" stāvokļiem un daļa uz "nemarķētajiem" stāvokļiem.

- $|\psi_g\rangle$ - normalizēta "marķēto" stāvokļu apakštelpa,
- $|\psi_b\rangle$ - normalizēta "nemarķēto" stāvokļu apakštelpa.

Tad

$$
|\psi\rangle = \sqrt{p}\,|\psi_g\rangle + \sqrt{1-p}\,|\psi_b\rangle,
$$

kur $p$ ir varbūtība, ka mērījumā iegūstam kādu marķēto stāvokli.

Ieviešam leņķi $\theta$, tā lai

$$
\sin^2\theta = p,
\qquad 0<\theta\le \frac{\pi}{2}.
$$

Tad varam rakstīt:

$$
|\psi\rangle = \sin\theta\,|\psi_g\rangle + \cos\theta\,|\psi_b\rangle.
$$

Šī ir 2-dimensionāla ģeometrija plaknē, ko veido $|\psi_g\rangle$ un $|\psi_b\rangle$.

## Refleksijas

Amplitude amplification balstās uz diviem refleksijas operatoriem.

### Oracle refleksija

Definējam fāzes orākulu:

$$
S_f |x\rangle = (-1)^{f(x)} |x\rangle.
$$

Tas ir atbildīgs par to, lai
- "marķētajiem" stāvokļiem fāze kļūst $-1$,
-  pārējiem stāvokļiem nekas nemainās.

### Refleksija ap sākuma stāvokli

Definējam refleksiju ap $|0\rangle$:

$$
S_0 = I - 2|0\rangle\langle 0|.
$$

Refleksiju ap $|\psi\rangle = A|0\rangle$ var realizēt kā:

$$
S_\psi = A S_0 A^\dagger.
$$

Definējam operatoru:

$$
Q = -A S_0 A^\dagger S_f.
$$

Tas ir divu refleksiju salikums.  
Divu refleksiju kompozīcija ģeometriski atbilst rotācijai.

## Rotācijas interpretācija

Sākotnējais stāvoklis:

$$
|\psi\rangle
=
\sin(\theta)\,|\psi_{\text{good}}\rangle
+
\cos(\theta)\,|\psi_{\text{bad}}\rangle.
$$

Pēc $k$ iterācijām iegūstam:

$$
Q^k |\psi\rangle
=
\sin\big((2k+1)\theta\big)\,|\psi_{\text{good}}\rangle
+
\cos\big((2k+1)\theta\big)\,|\psi_{\text{bad}}\rangle.
$$

Tātad varbūtība, ka tiks nomērīts marķēts stāvoklis pēc $k$ iterācijām ir:

$$
p_k = \sin^2\big((2k+1)\theta\big).
$$

Maksimālā varbūtība tiek sasniegta, ja:

$$
(2k+1)\theta \approx \frac{\pi}{2}.
$$

No tā iegūst aptuveno optimālo iterāciju skaitu:

$$
k \approx \left\lfloor \frac{\pi}{4\theta} - \frac{1}{2} \right\rfloor.
$$

Ja $k$ ir pārāk liels, tad $(2k+1)\theta$ pārsniedz $\pi/2$ un varbūtība sāk kristies.  

## Saistība ar Grovera algoritmu

Grovera meklēšana ir īpašs amplitūdu pastiprināšanas gadījums, kur:

- $A$ sagatavo vienmērīgu superpozīciju,
- $S_f$ atzīmē meklēto elementu,
- $Q$ tiek atkārtots optimālu reižu skaitu.


In [9]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

# Marķētais stāvoklis ir <101|

qc = QuantumCircuit(3, 3)

# Oracle realizācija  (vispārīga fāzes apgriešana konkrētam bitstringam)
def phase_flip(qc, qubits, bitstring):
    q0, q1, q2 = qubits
    bits = [int(b) for b in bitstring]

    # Pārveido mērķa stāvokli uz |111> (kur bit = 0, uzliek X)
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

    # CCZ realizācija caur H + CCX + H uz pēdējā kubita
    qc.h(q2)
    qc.ccx(q0, q1, q2)
    qc.h(q2)

    # Atceļ iepriekšējos X
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

# sagatavo brīvi izvēlētu nevienmērīga superpozīcija
qc.ry(0.9, 0)
qc.ry(1.7, 1)
qc.ry(1.2, 2)

# Iterāciju skaits
num_iterations = 2

for i in range(num_iterations):

    # Sf: oracle 
    phase_flip(qc, [0, 1, 2], "101")

    # A†: sagatavošanas inversais (apgriezta secība un negatīvi leņķi)
    qc.ry(-1.2, 2)
    qc.ry(-1.7, 1)
    qc.ry(-0.9, 0)

    # S0: fāzes inversija tikai |000>
    
    qc.x([0, 1, 2])

    # CCZ uz |111>
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

    qc.x([0, 1, 2])   # atpakaļ uz oriģinālo bāzi

    # A: atkal uz priekšu
    qc.ry(0.9, 0)
    qc.ry(1.7, 1)
    qc.ry(1.2, 2)

qc.measure([0, 1, 2], [0, 1, 2])

print(qc.draw())

sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=2000).result()
counts = result.get_counts()

print("Mērījumu sadalījums:", counts)

     ┌─────────┐          ┌──────────┐   ┌───┐                   ┌───┐»
q_0: ┤ Ry(0.9) ├───────■──┤ Ry(-0.9) ├───┤ X ├────────────────■──┤ X ├»
     ├─────────┤┌───┐  │  └──┬───┬───┘┌──┴───┴───┐┌───┐       │  ├───┤»
q_1: ┤ Ry(1.7) ├┤ X ├──■─────┤ X ├────┤ Ry(-1.7) ├┤ X ├───────■──┤ X ├»
     ├─────────┤├───┤┌─┴─┐   ├───┤    ├──────────┤├───┤┌───┐┌─┴─┐├───┤»
q_2: ┤ Ry(1.2) ├┤ H ├┤ X ├───┤ H ├────┤ Ry(-1.2) ├┤ X ├┤ H ├┤ X ├┤ H ├»
     └─────────┘└───┘└───┘   └───┘    └──────────┘└───┘└───┘└───┘└───┘»
c: 3/═════════════════════════════════════════════════════════════════»
                                                                      »
«     ┌─────────┐                     ┌──────────┐   ┌───┐                   »
«q_0: ┤ Ry(0.9) ├──────────────────■──┤ Ry(-0.9) ├───┤ X ├────────────────■──»
«     ├─────────┤   ┌───┐          │  └──┬───┬───┘┌──┴───┴───┐┌───┐       │  »
«q_1: ┤ Ry(1.7) ├───┤ X ├──────────■─────┤ X ├────┤ Ry(-1.7) ├┤ X ├───────■──»
«     └──┬───┬──┘┌──┴───┴──┐┌───┐┌─┴